In [1]:
from PIL import Image
import numpy as np
img = Image.open("/home/ubuntu/fonts/rendered_fonts/ABeeZee-Italic/B_upper.png")
img = np.array(img)
img.mean()


213.74444580078125

In [13]:
def is_dark(pil_image, threshold=0.3):
    """
    Determines if a font image is dark enough based on pixel intensity.
    
    Parameters:
    - pil_image: PIL Image object containing the rendered font
    - threshold: Float between 0 and 1. Lower values mean darker fonts will pass.
                 For example, 0.8 means the image must be at least 20% black.
    
    Returns:
    - bool: True if the font is dark enough (passes the threshold), False otherwise
    """
    # Convert to grayscale if it's not already
    if pil_image.mode != 'L':
        pil_image = pil_image.convert('L')
    
    # Convert to numpy array for calculations
    img_array = np.array(pil_image)
    
    # Calculate mean intensity (0 = black, 255 = white)
    mean_intensity = np.mean(img_array) / 255.0
    
    # Return True if the image is dark enough (mean intensity is below threshold)
    return mean_intensity < threshold

In [21]:
path = "/home/ubuntu/fonts/rendered_fonts"

import os
from PIL import Image

def check_folder_for_dark_images(args):
    """
    Helper function to check a single folder for dark images.
    
    Parameters:
    - args: Tuple containing (base_path, font_folder, threshold)
    
    Returns:
    - str or None: Folder name if it contains dark images, None otherwise
    """
    base_path, font_folder, threshold = args
    font_folder_path = os.path.join(base_path, font_folder)
    
    # Skip if not a directory
    if not os.path.isdir(font_folder_path):
        return None
    
    # Check each image in the folder
    for image_file in os.listdir(font_folder_path):
        if not image_file.endswith(('.png', '.jpg', '.jpeg')):
            continue
            
        image_path = os.path.join(font_folder_path, image_file)
        
        try:
            # Open the image
            img = Image.open(image_path)
            
            # Check if the image is dark
            if is_dark(img, threshold):
                print(f"Found dark font: {font_folder}, {image_path}")
                return font_folder
        except Exception as e:
            print(f"Error processing {image_path}: {e}")
    
    return None

def find_dark_font_folders(base_path, threshold=0.4):
    """
    Scans through font folders and identifies those containing dark images.
    Uses multiprocessing for faster processing.
    
    Parameters:
    - base_path: Path to the directory containing font folders
    - threshold: Threshold for determining if an image is dark
    
    Returns:
    - list: Folders that contain at least one dark image
    """
    import multiprocessing as mp
    
    # Get all font folders
    font_folders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))]
    
    # Prepare arguments for the worker function
    args = [(base_path, folder, threshold) for folder in font_folders]
    
    # Use multiprocessing to check folders in parallel
    with mp.Pool() as pool:
        results = pool.map(check_folder_for_dark_images, args)
    
    # Filter out None values and return the list of folders with dark images
    folders_to_remove = [folder for folder in results if folder is not None]
    
    return folders_to_remove

# Example usage
dark_folders = find_dark_font_folders(path)
print(f"Found {len(dark_folders)} folders with dark images")
print(dark_folders[:10] if dark_folders else "No dark folders found")


Found dark font: Boldonse-Regular, /home/ubuntu/fonts/rendered_fonts/Boldonse-Regular/M_upper.png
Found dark font: ZillaSlabHighlight-Regular, /home/ubuntu/fonts/rendered_fonts/ZillaSlabHighlight-Regular/M_upper.png
Found 2 folders with dark images
['Boldonse-Regular', 'ZillaSlabHighlight-Regular']


In [26]:
font = 'ZillaSlabHighlight-Regular'
grid = Image.open(f"{path}/{font}/grid.png")

In [30]:
import shutil
shutil.rmtree(f"{path}/{font}")

In [10]:
path = "/home/ubuntu/fonts/rendered_fonts/NotoSerifDisplay[wdth,wght]/9_digit.png"

img = Image.open(path)
is_dark(img)

True